# Prodigy_Fusion_Study — R26 (6 esperimenti, fusione TOP win R25)

**Obiettivo**: verificare se i 3 fattori indipendenti che migliorano T-tracking (A4, B1, C1) sono ORTOGONALI e si sommano linearmente, o se ci sono interazioni.

## Contesto e ipotesi

Da R25 (18 run ablation, scenario mixed, Prodigy V08) abbiamo trovato 3 fattori che ognuno migliora T_tracking_corr senza danneggiare val_total:

| Asse R25 | Fattore | ΔT_corr | Δval |
|---|---|---:|---:|
| A4 | max_delay 6→18 (memoria sinaptica) | +0.090 | -0.015 |
| B1 | lambda_T_aux 0→0.1 (supervisione T) | +0.147 | -0.006 |
| C1 | lambda_sr 0.5→0 (no spike reg) | +0.088 | -0.014 |
| **somma teorica** | A4+B1+C1 | **+0.325** | **-0.035** |

Baseline R25_A1 (replica mixed V08): val=0.195, T_corr=0.353.
**Best atteso F1 (A4+B1+C1)**: val~0.16, T_corr~0.68 se sommano linearmente. Realisticamente 0.55-0.62 per non-linearità.

## 6 esperimenti (combinatoria + controllo)

| Run | max_delay | lambda_T_aux | lambda_sr | bit_shift | epochs | Note |
|---|---:|---:|---:|---:|---:|---|
| **F0_baseline_replica** | 6 | 0.0 | 0.5 | 3 | 10 | identico R25_A1, sanity replica |
| **F1_TRIPLE_win** | 18 | 0.1 | 0.0 | 3 | 10 | TOP candidato: tutti 3 win |
| F2_A4_B1 | 18 | 0.1 | 0.5 | 3 | 10 | memoria + T_aux, no sr_off |
| F3_B1_C1 | 6 | 0.1 | 0.0 | 3 | 10 | T_aux + no_sr (no memoria) |
| F4_A4_C1 | 18 | 0.0 | 0.0 | 3 | 10 | memoria + no_sr (no T_aux) |
| F5_TRIPLE_short | 18 | 0.1 | 0.0 | 3 | 5 | F1 + meno training (E asse suggerisce) |

**Totale: 6 run × ~10 min = ~1h Azure** (vs 3-4h R25).

## Tag naming

`R26_F<i>_<descr>` es. `R26_F1_TRIPLE_win`, `R26_F5_TRIPLE_short`.

## Output diagnostici

Tutti i plot R25 (G1-G18) + le metriche tracking sui CSV. Focus analisi:
- **Se F1 ≈ baseline + 0.30 su T_corr** → effetti SOMMANO (sweet spot trovato)
- **Se F1 < F2+F3+F4 marginali** → interazione NEGATIVA tra fattori
- **Se F5 batte F1** → meno training è ancora meglio (E asse confermato)

## Reference

- `results/Prodigy_Study/Ablation_R25/_aggregate_full.csv` — base per delta vs baseline
- `Prodigy_Ablation_Study_R25.ipynb` — studio causale che ha identificato i 3 win

In [ ]:
# Cell 1 -- Bootstrap + ENV check (richiamo da R25, R25 changes già verificati)
import sys, os, subprocess
import importlib.util as _imu

for pkg in ['prodigyopt', 'pandas', 'matplotlib']:
    if _imu.find_spec(pkg) is None:
        print(f'  installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for f in ['train.py', 'core/network.py', 'utils/plot_diagnostics.py',
          'results/Prodigy_Study/Ablation_R25/_aggregate_full.csv']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

# Verifica R25 changes attive (per le 3 CLI + colonne)
sys.path.insert(0, '.')
if 'train' in sys.modules: del sys.modules['train']
from train import pinn_loss, CSVLogger, BatchCSVLogger
for c in ['val_T_tracking_corr', 'val_T_pred_mean']:
    assert c in CSVLogger.COLS
for c in ['gn_out_fc_T', 'gn_decoded_T', 'grad_dir_T']:
    assert c in BatchCSVLogger.COLS
import inspect
assert 'lam_T_aux' in inspect.signature(pinn_loss).parameters
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--lambda_T_aux', '--cf_max_delay', '--cf_bit_shift']:
    assert flag in help_txt, f'MISSING: {flag}'
print('[OK] R25 changes presenti (3 CLI + 11+16 colonne CSV)')

br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
print(f'\n[GIT] branch: {br}')
assert br == 'Prodigy_Deep_Study'
print('\nENV check passed.')

In [ ]:
# Cell 2 -- 6 EXPERIMENTS (BASELINE + 5 fusion)
BASELINE = {
    'optimizer': 'prodigy', 'lr': 1.0,
    'd_coef': 1.0, 'd0': 1e-6, 'growth_rate': 'inf',
    'scheduler': 'cosine_no_restart',
    'scenario': 'mixed',
    'scenario_mix': 'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',
    'cut_in_ratio': 0.0,
    'cache_path': 'data/cache_1500_mixed_cut0.0_ou0.0.pt',
    'hidden_size': 32, 'rank': 8,
    'seq_len': 50, 'max_delay': 6, 'bit_shift': 3,
    'epochs': 10, 'max_steps_per_epoch': 100,
    'lambda_sr': 0.5, 'lambda_T_aux': 0.0,
}

def _exp(tag, desc, **overrides):
    e = dict(BASELINE)
    e.update(overrides)
    e['tag'] = tag
    e['desc'] = desc
    return e

EXPERIMENTS = [
    # Baseline replica — sanity (atteso val~0.195, T_corr~0.35)
    _exp('R26_F0_baseline_replica', 'Baseline replica (sanity, = R25_A1)'),

    # F1: TRIPLE WIN — combinazione completa
    _exp('R26_F1_TRIPLE_win',
         'A4+B1+C1: max_delay=18, T_aux=0.1, sr=0 (TOP candidato)',
         max_delay=18, lambda_T_aux=0.1, lambda_sr=0.0),

    # Coppie (controllo per isolare interazioni)
    _exp('R26_F2_A4_B1',
         'A4+B1: max_delay=18, T_aux=0.1 (no sr_off)',
         max_delay=18, lambda_T_aux=0.1),
    _exp('R26_F3_B1_C1',
         'B1+C1: T_aux=0.1, sr=0 (no memoria)',
         lambda_T_aux=0.1, lambda_sr=0.0),
    _exp('R26_F4_A4_C1',
         'A4+C1: max_delay=18, sr=0 (no T_aux)',
         max_delay=18, lambda_sr=0.0),

    # F5: TRIPLE + meno training (asse E suggerisce)
    _exp('R26_F5_TRIPLE_short',
         'F1 + epochs=5 (early stop, asse E)',
         max_delay=18, lambda_T_aux=0.1, lambda_sr=0.0, epochs=5),
]

EXPECTED_N = 6
assert len(EXPERIMENTS) == EXPECTED_N
print(f'Total: {len(EXPERIMENTS)} run')
for e in EXPERIMENTS:
    print(f'  {e["tag"]:<30} md={e["max_delay"]:>2} T_aux={e["lambda_T_aux"]:>4} sr={e["lambda_sr"]:>4} ep={e["epochs"]:>2}')

In [ ]:
# Cell 3 -- Cache (riuso da R25)
import os
cache = BASELINE['cache_path']
assert os.path.isfile(cache), f'Cache mixed mancante: {cache} — generala dal notebook R25 cell 3'
sz = os.path.getsize(cache) / 1024 / 1024
print(f'[OK] {cache} ({sz:.1f} MB)')

In [ ]:
# Cell 4 -- PRE-FLIGHT (statica + empirica, smoke veloce)
import torch, inspect, sys, numpy as np
sys.path.insert(0, '.')
if 'core.network' in sys.modules: del sys.modules['core.network']
from core.network import build_model

print('=== Pre-flight 1: fix statici ===')
torch.manual_seed(42)
m = build_model('baseline', max_delay=18, bit_shift=3)
assert sum(p.numel() for p in m.parameters()) == 864
print(f'  baseline+max_delay=18: 864 params OK')

print('\n=== Pre-flight 2: smoke 1 ep × 3 step, R26 config TRIPLE ===')
import subprocess as sp, shutil, time

def _robust_rmtree(path, max_retries=3):
    """NFS-safe rmtree con retry + backoff (Azure cluster)."""
    for attempt in range(max_retries):
        if not os.path.isdir(path):
            return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path):
            return True
        time.sleep(0.5 * (attempt + 1))
    shutil.rmtree(path, ignore_errors=True)
    return not os.path.isdir(path)

# FIX NFS: usa tag UNIVOCO con timestamp per evitare race condition rmtree↔makedirs.
# Non puliamo prima del train.py (la stale metadata NFS confonde os.makedirs).
# Cleanup alla fine — se fallisce non blocchiamo (ignore_errors=True).
tag = f'_R26_PREFLIGHT_{int(time.time())}'

r = sp.run([sys.executable, 'train.py',
            '--training_method', 'baseline',
            '--epochs', '1', '--max_steps_per_epoch', '3',
            '--batch_size', '4', '--seq_len', '30',
            '--data_cache', BASELINE['cache_path'],
            '--n_train', '16', '--n_val', '8',
            '--cf_max_delay', '18', '--cf_bit_shift', '3',
            '--lambda_T_aux', '0.1', '--lambda_sr', '0.0',
            '--tag', tag],
           capture_output=True, text=True, encoding='utf-8', errors='replace')
if r.returncode != 0:
    print('STDOUT:', r.stdout[-1500:])
    print('STDERR:', r.stderr[-1500:])
assert r.returncode == 0, f'train.py exit {r.returncode}'

import pandas as pd
bdf = pd.read_csv(f'checkpoints/{tag}/training_batch_log.csv')
for c in ('loss_T_aux', 'gn_out_fc_T', 'grad_dir_T'):
    assert bdf[c].notna().any(), f'col {c} tutta NaN'
# Verifica loss_T_aux > 0 (lambda_T_aux=0.1 attivo)
assert bdf['loss_T_aux'].iloc[-1] > 0, 'loss_T_aux non attivo'
print(f'  Smoke R26 config TRIPLE OK: {len(bdf)} batch, loss_T_aux={bdf["loss_T_aux"].iloc[-1]:.4f}')

# Cleanup finale (best-effort, no asserzioni)
_robust_rmtree(f'checkpoints/{tag}')

print('\n[OK] Pre-flight superato. Procedere con sweep 6 run R26.')

In [ ]:
# Cell 5 -- RUN sweep 6 run con PUSH per-run (riusa pattern R24F/R25)
import subprocess, sys, time, os, shutil, glob, datetime
import pandas as pd

SKIP_IF_EXISTS = True
RESULTS_DIR = 'results/Prodigy_Study/Fusion_R26'
os.makedirs(RESULTS_DIR, exist_ok=True)
_TMP_MSG = '/tmp/r26_msg.txt' if os.path.isdir('/tmp') else 'r26_msg.txt'

def _robust_rmtree(path, max_retries=3):
    """Su Azure (NFS) rmtree fallisce con 'Directory not empty' anche dopo aver
    rimosso i contenuti — race con metadati NFS. Retry con backoff."""
    for attempt in range(max_retries):
        if not os.path.isdir(path):
            return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path):
            return True
        time.sleep(0.5 * (attempt + 1))
    shutil.rmtree(path, ignore_errors=True)
    return not os.path.isdir(path)

def _build_cli(e):
    return [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', str(e['epochs']),
        '--max_steps_per_epoch', str(e['max_steps_per_epoch']),
        '--batch_size', '8', '--val_batch_size', '32',
        '--seq_len', str(e['seq_len']),
        '--cf_hidden_size', str(e['hidden_size']),
        '--cf_rank', str(e['rank']),
        '--cf_max_delay', str(e['max_delay']),
        '--cf_bit_shift', str(e['bit_shift']),
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', str(e['lambda_sr']),
        '--lambda_T_aux', str(e['lambda_T_aux']),
        '--scenario_mix', e['scenario_mix'],
        '--cut_in_ratio', str(e['cut_in_ratio']),
        '--noise_scale', '0.0', '--po2_enabled', '1',
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--data_cache', e['cache_path'],
        '--optimizer', e['optimizer'],
        '--lr', str(e['lr']), '--max_lr', str(e['lr']),
        '--scheduler', e['scheduler'],
        '--prodigy_betas', '0.9,0.99',
        '--prodigy_d_coef', str(e['d_coef']),
        '--prodigy_d0', str(e['d0']),
        '--prodigy_weight_decay', '0.01',
        '--prodigy_use_bias_correction', '1',
        '--prodigy_safeguard_warmup', '1',
        '--prodigy_growth_rate', str(e['growth_rate']),
        '--tag', e['tag']]

def _push_run(e):
    tag = e['tag']
    src = f'checkpoints/{tag}'
    dst = f'{RESULTS_DIR}/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN push] {src} mancante')
        return False
    _robust_rmtree(dst)  # NFS-safe cleanup
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')
    val_str = ''
    log_path = f'{dst}/training_log.csv'
    if os.path.isfile(log_path):
        try:
            edf = pd.read_csv(log_path)
            if len(edf) > 0:
                bi = int(edf.val_total.idxmin())
                tcorr = edf.get('val_T_tracking_corr', pd.Series([float('nan')]))
                tc = tcorr.iloc[bi] if len(tcorr) > bi else float('nan')
                val_str = f'best val={edf.val_total.min():.4f} T_corr={tc:.3f} (E{bi+1}/{len(edf)})'
        except Exception as ex:
            val_str = f'(log read failed: {ex})'
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (f'results (R26 Fusion): {tag} ({ts})\n\n{val_str}\n'
           f'desc={e["desc"]}\nBranch Prodigy_Deep_Study\n')
    with open(_TMP_MSG, 'w', encoding='utf-8') as fp:
        fp.write(msg)
    try:
        subprocess.run(['git','add',dst], check=True, capture_output=True)
        r = subprocess.run(['git','commit','-F',_TMP_MSG], capture_output=True, text=True)
        if r.returncode != 0:
            if 'nothing to commit' in r.stdout or 'nothing to commit' in r.stderr:
                return True
            print(f'   [push commit fail] {r.stderr[-300:]}'); return False
        subprocess.run(['git','pull','--no-rebase','--no-edit','origin','Prodigy_Deep_Study'],
                       capture_output=True, text=True)
        r2 = subprocess.run(['git','push','origin','Prodigy_Deep_Study'], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'   [push fail] {r2.stderr[-300:]}'); return False
        print(f'   [push OK]')
        return True
    finally:
        try: os.remove(_TMP_MSG)
        except: pass

run_results = []
t_start = time.time()
total = len(EXPERIMENTS)
for i, e in enumerate(EXPERIMENTS, 1):
    tag = e['tag']
    dst_log = f'{RESULTS_DIR}/{tag}/training_log.csv'
    if SKIP_IF_EXISTS and os.path.isfile(dst_log):
        try:
            edf = pd.read_csv(dst_log)
            v_str = f'val={edf.val_total.min():.4f}'
        except: v_str = 'unreadable'
        print(f'\n[{i}/{total}] [SKIP] {tag}: {v_str}')
        run_results.append({'tag': tag, 'status':'skipped'}); continue
    print(f'\n{"="*78}\n[{i}/{total}] {tag}: {e["desc"]}\n{"="*78}')
    t0 = time.time()
    r = subprocess.run(_build_cli(e), capture_output=False)
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    el_tot = time.time() - t_start
    done_now = sum(1 for x in run_results if x['status']!='skipped') + 1
    eta_min = (el_tot / max(done_now,1)) * (total - i) / 60
    print(f'\n[{i}/{total}] {tag} -> {status}  ({el/60:.1f}min)  ETA={eta_min:.0f}min')
    pushed = _push_run(e)
    run_results.append({'tag': tag, 'status': status, 'pushed': pushed})

print(f'\n{"="*78}\nSWEEP DONE in {(time.time()-t_start)/60:.0f}min')
for r in run_results:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<45} {r['status']:<10} push={r.get('pushed','N/A')}")

In [ ]:
# Cell 6 -- Aggregate + linearity test (somma effetti R25 vs realtà R26)
import pandas as pd, os
RESULTS_DIR = 'results/Prodigy_Study/Fusion_R26'

rows = []
for e in EXPERIMENTS:
    log = f'{RESULTS_DIR}/{e["tag"]}/training_log.csv'
    if not os.path.isfile(log):
        rows.append({'tag': e['tag'], 'status':'MISSING'}); continue
    df = pd.read_csv(log)
    if len(df) == 0:
        rows.append({'tag': e['tag'], 'status':'EMPTY'}); continue
    bi = int(df['val_total'].idxmin())
    r = {'tag': e['tag'], 'desc': e['desc'], 'status':'OK', 'best_ep': bi+1,
         'val_total_best': float(df['val_total'].iloc[bi]),
         'val_data_best':  float(df['val_data'].iloc[bi]),
         'spike_rate_best': float(df['spike_rate'].iloc[bi]),
    }
    for c in ('val_T_tracking_corr','val_T_pred_mean','val_T_intra_std',
              'val_v0_pred_mean','val_s0_pred_mean','val_a_pred_mean','val_b_pred_mean'):
        if c in df.columns: r[f'{c}_best'] = float(df[c].iloc[bi])
    rows.append(r)

df_R26 = pd.DataFrame(rows)
ok = df_R26[df_R26.status=='OK'].copy()

# Baseline R26 = F0 (replica)
f0 = ok[ok.tag == 'R26_F0_baseline_replica']
if len(f0) > 0:
    base_val = float(f0.val_total_best.iloc[0])
    base_Tcorr = float(f0.val_T_tracking_corr_best.iloc[0])
    print(f'R26_F0 baseline: val={base_val:.4f}, T_corr={base_Tcorr:.3f}')
    # R25 reference
    print(f'R25_A1 reference (atteso): val=0.195, T_corr=0.353')
    print()
    ok['dval'] = ok.val_total_best - base_val
    ok['dTcorr'] = ok.val_T_tracking_corr_best - base_Tcorr
    
    print('=== R26 RESULTS (vs F0 baseline replica) ===')
    print(ok[['tag','val_total_best','dval','val_T_tracking_corr_best','dTcorr','spike_rate_best','best_ep']].round(4).to_string(index=False))

    # Linearity test: confronto F1 (triple) vs somma effetti R25
    print()
    print('='*70)
    print('LINEARITY TEST: F1 (TRIPLE) vs somma R25 effetti')
    print('='*70)
    r25_predictions = {'dval': -0.015 + -0.006 + -0.014,
                       'dTcorr': 0.090 + 0.147 + 0.088}
    print(f'R25 sum predicted: dval={r25_predictions["dval"]:+.4f} | dTcorr={r25_predictions["dTcorr"]:+.3f}')
    f1 = ok[ok.tag == 'R26_F1_TRIPLE_win']
    if len(f1) > 0:
        f1_dval = float(f1.dval.iloc[0])
        f1_dTcorr = float(f1.dTcorr.iloc[0])
        print(f'R26 F1 measured:   dval={f1_dval:+.4f} | dTcorr={f1_dTcorr:+.3f}')
        ratio_T = f1_dTcorr / r25_predictions['dTcorr'] if r25_predictions['dTcorr'] != 0 else float('nan')
        print(f'\nLinearity ratio T_corr: {ratio_T*100:.0f}% '
              f'({"linear sum" if abs(ratio_T-1) < 0.2 else "NON-linear: " + ("saturating" if ratio_T < 1 else "super-linear")})')

ok.to_csv(f'{RESULTS_DIR}/_aggregate_r26.csv', index=False)
print(f'\n[saved] {RESULTS_DIR}/_aggregate_r26.csv')

In [ ]:
# Cell 7 -- Plot comparison F0 vs F1 (TRIPLE WIN candidate)
from IPython.display import Image, display, Markdown
import os
RESULTS_DIR = 'results/Prodigy_Study/Fusion_R26'

for tag, label in [
    ('R26_F0_baseline_replica', 'F0 BASELINE replica'),
    ('R26_F1_TRIPLE_win', 'F1 TRIPLE WIN candidate'),
    ('R26_F5_TRIPLE_short', 'F5 TRIPLE + short training'),
]:
    display(Markdown(f'## {label} ({tag})'))
    for plot in ('G7_violin_params.png', 'G13_traj_highway.png',
                 'G16_grad_raw_per_ch.png', 'G18_grad_direction_per_ch.png'):
        p = f'{RESULTS_DIR}/{tag}/plots/{plot}'
        if os.path.isfile(p):
            display(Image(p, width=900))
        else:
            print(f'  [missing] {p}')

In [ ]:
# Cell 8 -- Decision summary + raccomandazione
from IPython.display import display, Markdown

if (df_R26.status=='OK').sum() == 0:
    print('Nessun run OK — saltare riepilogo.')
else:
    ok = df_R26[df_R26.status=='OK'].copy()
    best_T = ok.sort_values('val_T_tracking_corr_best', ascending=False).iloc[0]
    best_v = ok.sort_values('val_total_best').iloc[0]
    
    display(Markdown(f'''
## R26 Fusion — Conclusioni

### Best T-tracking
**{best_T.tag}**: T_corr = {best_T.val_T_tracking_corr_best:.3f} | val = {best_T.val_total_best:.4f}
({best_T.desc})

### Best val_total
**{best_v.tag}**: val = {best_v.val_total_best:.4f} | T_corr = {best_v.val_T_tracking_corr_best:.3f}
({best_v.desc})

### Interpretazione delle 4 combinazioni controllo (F2, F3, F4 vs F1)
Se F1 > max(F2, F3, F4) → effetti SOMMANO (success).
Se F1 ≈ max delle coppie → c'è saturazione.
Se F1 < max delle coppie → interazione NEGATIVA (rara).
    '''))
    
    sub = ok[ok.tag.isin(['R26_F1_TRIPLE_win','R26_F2_A4_B1','R26_F3_B1_C1','R26_F4_A4_C1'])]
    if len(sub) >= 2:
        display(sub[['tag','val_total_best','val_T_tracking_corr_best','spike_rate_best','best_ep']].round(4))